# Audit initial du prototype RAG NBA
 ---

**Parcours :** OpenClassrooms — Data Scientist / Machine Learning  
**Projet :** Projet 10 — Évaluez les performances d'un LLM  
**Date :** 2026-06-04

---

## Objectif

Ce notebook constitue l'audit initial du prototype RAG fourni dans l'énoncé : un assistant NBA basé sur Mistral, un index vectoriel FAISS et une interface Streamlit.

L'objectif est de vérifier, de façon reproductible, que chaque composant du prototype fonctionne : dépendances, données, index FAISS, API Mistral, recherche vectorielle et pipeline RAG complet. Le notebook sert aussi à documenter les limites observées avant toute amélioration.

## Prototype audité

- `MistralChat.py` — application Streamlit de chat RAG.
- `indexer.py` — indexation des documents : OCR, embeddings, FAISS.
- `utils/` — configuration, chargement des données et gestion du vector store.
- `inputs/` — 4 PDF Reddit et 1 fichier Excel de statistiques NBA.
- `vector_db/` — index FAISS généré localement par `python indexer.py`, non versionné.

## Périmètre de cet audit

Cet audit est volontairement limité à une vérification fonctionnelle et qualitative du prototype fourni.

Le code applicatif n'est pas modifié dans ce notebook. Les questions de test sont choisies manuellement : elles ne constituent pas encore un jeu d'évaluation stable. La mesure objective avec RAGAS sera réalisée dans une étape suivante.

## Prérequis d'exécution

- Un fichier `.env` local contenant `MISTRAL_API_KEY=...`.
- Un index FAISS construit avec `python indexer.py`.
- L'environnement Python du projet activé.

La clé API ne doit jamais être affichée entièrement dans le notebook ni versionnée dans Git.

## Résumé exécutif

**Ce qui fonctionne ✅**

Le prototype démarre correctement, l'index FAISS est cohérent, les appels Mistral fonctionnent et la recherche vectorielle retourne des chunks pertinents. Le pipeline RAG répond sans planter aux questions NBA simples, aux questions bruitées et aux questions hors sujet.

**Limite principale ⚠️**

Le prototype n'est pas fiable pour les questions chiffrées. Il récupère des chunks proches, puis le LLM reformule ou complète avec ses connaissances générales au lieu de calculer réellement la réponse à partir des données. Le problème principal est donc la **faithfulness** : la réponse peut être plausible, mais non ancrée dans les sources.

**Conséquence pour le projet 🎯**

Cet audit justifie deux étapes clés du projet : utiliser RAGAS pour mesurer objectivement les réponses, puis ajouter un accès structuré aux données Excel via SQLite / SQL Tool pour traiter les questions numériques.

---
# Imports

In [1]:
import os, sys
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS : évite un crash OpenMP (faiss + torch/easyocr)
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
# Utilitaires d'affichage
from IPython.display import display, HTML

def ok(msg):  print(f"\033[92m ✅ {msg}\033[0m")
def ko(msg):  print(f"\033[91m ❌ {msg}\033[0m")
def warn(msg):print(f"\033[93m ⚠️  {msg}\033[0m")

---
# Dépendances

On vérifie d'abord que toutes les bibliothèques du prototype sont installées et importables, puis on relève leurs **versions** — utile pour repérer un SDK obsolète comme `mistralai < 1.0`.

In [3]:
import importlib

# easyocr est testé séparément (chargement lourd des modèles GPU/CPU)
deps = [
    ("streamlit",       "Interface web"),
    ("mistralai",       "SDK Mistral AI"),
    ("faiss",           "Index vectoriel FAISS"),
    ("langchain",       "Text splitter"),
    ("langchain_core",  "Langchain core"),
    ("dotenv",          "Chargement .env"),
    ("PyPDF2",          "Extraction texte PDF"),
    ("fitz",            "PyMuPDF (OCR)"),
    ("PIL",             "Pillow (images)"),
    ("pandas",          "Lecture Excel/CSV"),
    ("openpyxl",        "Moteur Excel"),
    ("docx",            "Lecture DOCX"),
    ("requests",        "Téléchargement HTTP"),
    ("numpy",           "Calcul vectoriel"),
    ("tqdm",            "Barre de progression"),
]

all_ok = True
for module, description in deps:
    try:
        importlib.import_module(module)
    except ImportError as e:
        ko(f"{module:<20} — {description}  →  {e}")
        all_ok = False

# easyocr vérifié sans import complet (évite le chargement des modèles)
try:
    import importlib.util
    spec = importlib.util.find_spec("easyocr")
    if not spec:
        ko(f"{'easyocr':<20} — OCR EasyOCR  →  module non trouvé")
        all_ok = False
except Exception as e:
    ko(f"easyocr check failed: {e}")
    all_ok = False

if all_ok:
    ok("Toutes les dépendances sont disponibles.")
else:
    ko("Certaines dépendances manquent. Relancez : pip install -r requirements.txt")

 ✅ Toutes les dépendances sont disponibles.


## Versions

In [4]:
import streamlit, mistralai, faiss, langchain, pandas, numpy
import importlib.metadata as _meta

def _ver(pkg):
    try: return _meta.version(pkg)
    except Exception: return "N/A"

versions = {
    "streamlit":  streamlit.__version__,
    "mistralai":  _ver("mistralai"),
    "langchain":  langchain.__version__,
    "pandas":     pandas.__version__,
    "numpy":      numpy.__version__,
    "faiss-cpu":  _ver("faiss-cpu"),
}
for lib, ver in versions.items():
    print(f"{lib:<12} {ver}")

streamlit    1.44.1
mistralai    0.4.2
langchain    0.3.23
pandas       2.2.3
numpy        2.4.6
faiss-cpu    1.10.0


  ⚠️  mistralai 0.4.2 est une version dépréciée (< 1.0). Le code fonctionne, mais la migration vers mistralai >= 1.0 est recommandée.

---
# Configuration et clé API

On charge la configuration centrale (`utils/config.py`) et on confirme que la clé API est présente.
> 🔒 La clé n'est **jamais** affichée en entier : seuls les premiers caractères et sa longueur sont montrés, juste pour prouver qu'elle est bien chargée.

In [27]:
import os
from dotenv import load_dotenv
load_dotenv()

# Clé API
api_key = os.getenv("MISTRAL_API_KEY")
if api_key:
    ok(f"MISTRAL_API_KEY trouvée — longueur {len(api_key)} caractères")
else:
    ko("MISTRAL_API_KEY absente ! Créez un fichier .env avec MISTRAL_API_KEY=...")

# Chargement de la config du projet
try:
    from utils.config import (
        MISTRAL_API_KEY, MODEL_NAME, EMBEDDING_MODEL,
        INPUT_DIR, VECTOR_DB_DIR, FAISS_INDEX_FILE,
        DOCUMENT_CHUNKS_FILE, CHUNK_SIZE, CHUNK_OVERLAP,
        EMBEDDING_BATCH_SIZE, SEARCH_K
    )
    ok("utils/config.py chargé avec succès")
    print(f"Modèle LLM         : {MODEL_NAME}")
    print(f"Modèle Embedding   : {EMBEDDING_MODEL}")
    print(f"Dossier inputs     : {INPUT_DIR}")
    print(f"Dossier vector_db  : {VECTOR_DB_DIR}")
    print(f"Taille chunk       : {CHUNK_SIZE} chars")
    print(f"Overlap chunk      : {CHUNK_OVERLAP} chars")
    print(f"Batch embeddings   : {EMBEDDING_BATCH_SIZE}")
    print(f"Recherche top-k    : {SEARCH_K}")
except ImportError as e:
    ko(f"Impossible de charger utils/config.py : {e}")

 ✅ MISTRAL_API_KEY trouvée — longueur 32 caractères
 ✅ utils/config.py chargé avec succès
Modèle LLM         : mistral-small-latest
Modèle Embedding   : mistral-embed
Dossier inputs     : inputs
Dossier vector_db  : vector_db
Taille chunk       : 1500 chars
Overlap chunk      : 150 chars
Batch embeddings   : 32
Recherche top-k    : 5


---
# Chargement des données

On contrôle la présence des fichiers sources, puis la **qualité du texte extrait**. Les PDF sont des captures d'écran lues par OCR, donc potentiellement bruitées.
> L'extraction OCR complète est longue (~3-5 min) ; elle est automatiquement ignorée si l'index FAISS existe déjà (`SKIP_OCR`).

## Présence des fichiers source

In [6]:
from pathlib import Path

input_path = Path(INPUT_DIR)
if not input_path.exists():
    ko(f"Le dossier '{INPUT_DIR}' n'existe pas.")
else:
    ok(f"Dossier '{INPUT_DIR}' présent.")
    files = list(input_path.rglob("*.*"))
    print(f"{len(files)} fichier(s) trouvé(s) :")
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f" - {f.name:<30} {size_kb:>7.1f} Ko  [{f.suffix}]")

 ✅ Dossier 'inputs' présent.
5 fichier(s) trouvé(s) :
 - Reddit 4.pdf                    2170.0 Ko  [.pdf]
 - Reddit 1.pdf                    3643.6 Ko  [.pdf]
 - Reddit 3.pdf                    8138.6 Ko  [.pdf]
 - Reddit 2.pdf                    5534.9 Ko  [.pdf]
 - regular NBA.xlsx                 241.3 Ko  [.xlsx]


## Extraction de texte
⚠️  Ce test lance l'OCR EasyOCR (chargement modèle ~500 Mo + traitement page par page)

Durée estimée : 3-5 minutes.

Passez SKIP_OCR = True pour ignorer ce test si l'index existe déjà.

In [7]:
SKIP_OCR = os.path.exists(FAISS_INDEX_FILE)  # Skip auto si l'index est déjà là

if SKIP_OCR:
    warn("Index FAISS déjà présent → test d'extraction OCR ignoré (SKIP_OCR=True).")
    warn("Mettez SKIP_OCR = False pour forcer le re-test d'extraction.")
    docs = []  # sera rechargé depuis l'index dans les cellules suivantes
else:
    import logging
    logging.disable(logging.WARNING)
    from utils.data_loader import load_and_parse_files
    docs = load_and_parse_files(INPUT_DIR)
    logging.disable(logging.NOTSET)

    if not docs:
        ko("Aucun document chargé. Vérifiez le dossier 'inputs/'.")
    else:
        ok(f"{len(docs)} document(s) chargé(s) et parsé(s)")
        print()
        print(f"  {'Source':<45} {'Chars':>8}  {'Méthode'}")
        print(f"  {'-'*45} {'-'*8}  {'-'*20}")
        for d in docs:
            src = d['metadata']['source']
            n = len(d['page_content'])
            ext = Path(d['metadata']['filename']).suffix
            method = "OCR (EasyOCR)" if ext == ".pdf" else "pandas" if ext in [".xlsx",".xls"] else ext
            flag = "  ⚠️  peu de texte" if n < 1000 else ""
            print(f"  {src:<45} {n:>8}  {method}{flag}")

 ⚠️  Index FAISS déjà présent → test d'extraction OCR ignoré (SKIP_OCR=True).
 ⚠️  Mettez SKIP_OCR = False pour forcer le re-test d'extraction.


## Qualité du texte extrait

In [8]:
if not docs:
    warn("Extraction ignorée (SKIP_OCR=True ou aucun document). Passez à la section 4.")
else:
    for d in docs:
        src = d['metadata']['source']
        text = d['page_content']
        lines = [l for l in text.split('\n') if l.strip()]

        noise_lines = sum(1 for l in lines if l.strip() in ('NaN', 'nan', '') or l.strip().isdigit())
        noise_ratio = noise_lines / max(len(lines), 1) * 100

        status = ok if noise_ratio < 20 else warn
        status(f"{src}")
        print(f"     {len(text):>7} chars | {len(lines)} lignes | bruit estimé : {noise_ratio:.0f}%")
        preview = [l[:80] for l in lines[:3]]
        for p in preview:
            print(f"         > {p}")

 ⚠️  Extraction ignorée (SKIP_OCR=True ou aucun document). Passez à la section 4.


---
# Index FAISS

On vérifie que l'index vectoriel et les chunks existent, se chargent, sont **cohérents** (autant de vecteurs que de chunks), puis on inspecte la taille des chunks et leur répartition par source.

## Fichier vector_db

In [9]:
import os

for filepath in [FAISS_INDEX_FILE, DOCUMENT_CHUNKS_FILE]:
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        ok(f"{filepath:<45} ({size_kb:.1f} Ko)")
    else:
        ko(f"{filepath} introuvable → relancez : python indexer.py")

 ✅ vector_db/faiss_index.idx                     (1208.0 Ko)
 ✅ vector_db/document_chunks.pkl                 (427.2 Ko)


## Chargement

In [10]:
from utils.vector_store import VectorStoreManager

mgr = VectorStoreManager()

if mgr.index is None or not mgr.document_chunks:
    ko("Index FAISS ou chunks non chargés. Relancez python indexer.py")
else:
    ok(f"Index chargé : {mgr.index.ntotal} vecteurs")
    ok(f"Chunks chargés : {len(mgr.document_chunks)}")
    
    if mgr.index.ntotal != len(mgr.document_chunks):
        ko(f"Désynchronisation : {mgr.index.ntotal} vecteurs ≠ {len(mgr.document_chunks)} chunks")
    else:
        ok("Cohérence index/chunks : OK")
    
    # Répartition par source
    print("Répartition des chunks par source :")
    sources = {}
    for c in mgr.document_chunks:
        s = c['metadata'].get('source', '?')
        sources[s] = sources.get(s, 0) + 1
    for s, n in sorted(sources.items(), key=lambda x: -x[1]):
        bar = '█' * (n // 2)
        print(f"     {n:>3} chunks  {bar:<20}  {s}")

2026-06-04 12:17:55,697 - INFO - Chargement de l'index Faiss depuis vector_db/faiss_index.idx...
2026-06-04 12:17:55,699 - INFO - Chargement des chunks depuis vector_db/document_chunks.pkl...
2026-06-04 12:17:55,701 - INFO - Index (302 vecteurs) et 302 chunks chargés.


 ✅ Index chargé : 302 vecteurs
 ✅ Chunks chargés : 302
 ✅ Cohérence index/chunks : OK
Répartition des chunks par source :
     143 chunks  ███████████████████████████████████████████████████████████████████████  regular NBA.xlsx (Feuille: Données NBA)
      42 chunks  █████████████████████  Reddit 3.pdf
      29 chunks  ██████████████        Reddit 2.pdf
      27 chunks  █████████████         regular NBA.xlsx (Feuille: Analyse)
      27 chunks  █████████████         regular NBA.xlsx (Feuille: Analyse Vide)
      18 chunks  █████████             Reddit 1.pdf
      11 chunks  █████                 Reddit 4.pdf
       4 chunks  ██                    regular NBA.xlsx (Feuille: Dictionnaire des données)
       1 chunks                        regular NBA.xlsx (Feuille: Equipe)


## Inspection des chunks

In [11]:
import pickle

with open(DOCUMENT_CHUNKS_FILE, 'rb') as f:
    chunks = pickle.load(f)

# Stats sur les tailles de chunks
sizes = [len(c['text']) for c in chunks]
print(f"Taille moyenne des chunks : {sum(sizes)/len(sizes):.0f} chars")
print(f" Min : {min(sizes)} chars")
print(f" Max : {max(sizes)} chars")

# Chunks trop courts (probablement des artefacts)
short = [c for c in chunks if len(c['text']) < 100]
if short:
    warn(f"{len(short)} chunk(s) très courts (< 100 chars) — probablement des artefacts d'OCR")
    for c in short[:3]:
        print(f"       id={c['id']} | src={c['metadata'].get('source','?')} | '{c['text'][:80]}'")
else:
    ok("Aucun chunk anormalement court.")


Taille moyenne des chunks : 1377 chars
 Min : 284 chars
 Max : 1499 chars
 ✅ Aucun chunk anormalement court.


## Exemple d'un chunk PDF

In [12]:
pdf_chunks = [c for c in chunks if '.pdf' in c['metadata'].get('source', '')]

for c in pdf_chunks[0:2]:
    print("----"*20)
    if '.pdf' in c['metadata'].get('source', ''):
        print(c['text'])

--------------------------------------------------------------------------------
12/06/2025 13:12
Which NBA team did not have home court advantage until the NBA Finals?
rInba
Accéder au contenu principal
Rechercher dans r/nba
Se connecter
rInba
ily a 12j
DonT012
Which NBA team did not have home court advantage until the NBA Finals?
In the NHL, the Edmonton Oilers just reached the NHL Finals. In their 3 rounds of play,
did not have home
court advantage:
played the Kings on the road. Then the Golden Knights on the road. Then finally , the
Stars on the road
will now host the Panthers at home for the Finals.
Which NBA team
experienced this? First 3 rounds all on the road. Finals at home
272
Partager
adidas_Ecom_Europe
Sponsorisé(e)
Inscris-toi à l'adiClub pour tenter de gagner une carte cadeau de 500 € avec ta Wishlist
GAGNE UNE CARTE
ADE
500
S'inscrire
adidas fr
Wishlist
ena
Rejoindre la conversation
Trier par
Meilleurs
Rechercher des commentaires
55555_55555
~12 j
Six teams have made the

## Exemple d'un chunk Excel

In [13]:
xls_chunks = [c for c in chunks if '.xlsx' in c['metadata'].get('source', '')]

for c in xls_chunks[:3]:
    print("----"*30)
    if '.xlsx' in c['metadata'].get('source', ''):
        print(c['text'])

------------------------------------------------------------------------------------------------------------------------
1     2    3   4   5   6     7     8    9     10    11        12   13    14   15   16    17    18    19    20   21   22   23   24   25    26   27   28    29      30      31      32    33      34         35     36     37    38        39    40    41    42      43    44    45  46  47  48  49  50  51  52  53
0                      Player  Team  Age  GP   W   L   Min   PTS  FGM   FGA   FG%  15:00:00  3PA   3P%  FTM  FTA   FT%  OREB  DREB   REB  AST  TOV  STL  BLK   PF    FP  DD2  TD3   +/-  OFFRTG  DEFRTG  NETRTG  AST%  AST/TO  AST RATIO  OREB%  DREB%  REB%  TO RATIO  EFG%   TS%  USG%    PACE   PIE  POSS NaN NaN NaN NaN NaN NaN NaN NaN
1     Shai Gilgeous-Alexander   OKC   26  76  63  13  34.2  2485  859  1657  51.9       160  433  37.5  600  669  89.8    68   312   380  486  182  129   76  167  4112    6    0  12.1   122.4   105.7    16.7  29.9    2.66       18.7    2.5 

---
# API Mistral

On teste les deux appels distants utilisés par le prototype : la **génération** (LLM) et les **embeddings**. Pour les embeddings, on vérifie en plus que des textes proches obtiennent une similarité élevée.

In [14]:
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage

try:
    client = MistralClient(api_key=MISTRAL_API_KEY)
    response = client.chat(
        model=MODEL_NAME,
        messages=[ChatMessage(role="user", content="Réponds uniquement avec le mot : OK")],
        temperature=0.0
    )
    reply = response.choices[0].message.content.strip()
    if "OK" in reply.upper():
        ok(f"LLM ({MODEL_NAME}) répond correctement : {reply!r}")
    else:
        warn(f"LLM répond mais pas comme attendu : {reply[:80]!r}")
    print(f"Tokens utilisés — prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens}")
except Exception as e:
    ko(f"Erreur LLM : {type(e).__name__} — {e}")

2026-06-04 12:17:56,235 - INFO - HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


 ✅ LLM (mistral-small-latest) répond correctement : 'OK'
Tokens utilisés — prompt: 24, completion: 2


## Embeddings

In [15]:
import numpy as np

try:
    test_texts = ["statistiques NBA", "rebonds domicile extérieur", "pourcentage à 3 points"]
    response = client.embeddings(model=EMBEDDING_MODEL, input=test_texts)
    embeddings = [e.embedding for e in response.data]
    dim = len(embeddings[0])
    ok(f"Embeddings générés : {len(embeddings)} vecteurs de dimension {dim}")
    
    # Vérifier la similarité cosinus entre textes proches
    import faiss as _faiss
    arr = np.array(embeddings, dtype='float32')
    _faiss.normalize_L2(arr)
    sim_01 = float(np.dot(arr[0], arr[1]))
    sim_02 = float(np.dot(arr[0], arr[2]))
    print(f"Similarité cosinus 'rebonds'  vs 'NBA stats'  : {sim_01:.3f}")
    print(f"Similarité cosinus '3 points' vs 'NBA stats'  : {sim_02:.3f}")
except Exception as e:
    ko(f"Erreur Embeddings : {type(e).__name__} — {e}")

2026-06-04 12:17:56,545 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"


 ✅ Embeddings générés : 3 vecteurs de dimension 1024
Similarité cosinus 'rebonds'  vs 'NBA stats'  : 0.609
Similarité cosinus '3 points' vs 'NBA stats'  : 0.730


## Recherche vectorielle

On interroge l'index avec quelques questions types et on observe les **scores de similarité** et les sources retournées

In [16]:
test_queries = [
    "Quel joueur a le meilleur pourcentage à 3 points ?",
    "Compare les rebonds à domicile et à l'extérieur",
    "Quelles équipes ont le plus de victoires cette saison ?",
    "Analyse des performances offensives",
]

for query in test_queries:
    print("---"*30)
    print(f"  🔍 {query}")
    results = mgr.search(query, k=3)
    if not results:
        ko("Aucun résultat retourné.")
    else:
        for i, r in enumerate(results):
            score = r['score']
            src = r['metadata'].get('source', '?')
            flag = "✅" if score > 75 else "⚠️ " if score > 60 else "❌"
            print(f"[{i+1}] {flag} Score: {score:.1f}%  |  {src}")
            print(f" {r['text'].strip()}...")

2026-06-04 12:17:56,793 - INFO - Recherche des 3 chunks les plus pertinents pour: 'Quel joueur a le meilleur pourcentage à 3 points ?'


------------------------------------------------------------------------------------------
  🔍 Quel joueur a le meilleur pourcentage à 3 points ?


2026-06-04 12:17:57,256 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:17:57,261 - INFO - 3 chunks pertinents trouvés.
2026-06-04 12:17:57,261 - INFO - Recherche des 3 chunks les plus pertinents pour: 'Compare les rebonds à domicile et à l'extérieur'


[1] ✅ Score: 80.2%  |  regular NBA.xlsx (Feuille: Dictionnaire des données)
 Dictionnaire des données                                                        Unnamed: 1
0                    Player                                                     Nom du joueur
1                      Team                               Équipe du joueur (code à 3 lettres)
2                       Age                                                     Âge du joueur
3                        GP                             Nombre de matchs joués (Games Played)
4                         W             Nombre de victoires de l'équipe lors des matchs joués
5                         L                                                Nombre de défaites
6                       Min                                 Minutes moyennes jouées par match
7                       PTS                               Points marqués en moyenne par match
8                       FGM                         Tirs réussis par match (Fiel

2026-06-04 12:17:57,508 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:17:57,521 - INFO - 3 chunks pertinents trouvés.
2026-06-04 12:17:57,522 - INFO - Recherche des 3 chunks les plus pertinents pour: 'Quelles équipes ont le plus de victoires cette saison ?'


[1] ⚠️  Score: 74.2%  |  regular NBA.xlsx (Feuille: Dictionnaire des données)
 41                     USG%                Usage Rate – % des actions utilisées par le joueur
42                     PACE                        Rythme de jeu (possessions par 48 minutes)
43                      PIE           Player Impact Estimate – évaluation globale de l’impact
44                     POSS                                Nombre total de possessions jouées...
[2] ⚠️  Score: 73.5%  |  regular NBA.xlsx (Feuille: Analyse Vide)
 129                       Analyse(s) complémentaire(s)                      NaN                          NaN                               NaN                                          NaN                                         NaN                NaN                               NaN...
[3] ⚠️  Score: 73.5%  |  regular NBA.xlsx (Feuille: Analyse)
 129                       Analyse(s) complémentaire(s)                      NaN                          NaN                 

2026-06-04 12:17:57,776 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:17:57,781 - INFO - 3 chunks pertinents trouvés.
2026-06-04 12:17:57,781 - INFO - Recherche des 3 chunks les plus pertinents pour: 'Analyse des performances offensives'


[1] ✅ Score: 80.8%  |  regular NBA.xlsx (Feuille: Equipe)
 Code Nom complet de l'équipe
0   ATL           Atlanta Hawks
1   BKN           Brooklyn Nets
2   BOS          Boston Celtics
3   CHA       Charlotte Hornets
4   CHI           Chicago Bulls
5   CLE     Cleveland Cavaliers
6   DAL        Dallas Mavericks
7   DEN          Denver Nuggets
8   DET         Detroit Pistons
9   GSW   Golden State Warriors
10  HOU         Houston Rockets
11  IND          Indiana Pacers
12  LAC    Los Angeles Clippers
13  LAL      Los Angeles Lakers
14  MEM       Memphis Grizzlies
15  MIA              Miami Heat
16  MIL         Milwaukee Bucks
17  MIN  Minnesota Timberwolves
18  NOP    New Orleans Pelicans
19  NYK         New York Knicks
20  OKC   Oklahoma City Thunder
21  ORL           Orlando Magic
22  PHI      Philadelphia 76ers
23  PHX            Phoenix Suns
24  POR  Portland Trail Blazers
25  SAC        Sacramento Kings
26  SAS       San Antonio Spurs
27  TOR         Toronto Raptors
28  UTA         

2026-06-04 12:17:57,984 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:17:57,986 - INFO - 3 chunks pertinents trouvés.


[1] ✅ Score: 75.8%  |  regular NBA.xlsx (Feuille: Analyse Vide)
 84   Analyse du top 15 des joueurs en nombre de points                      NaN                          NaN                               NaN                                          NaN                                         NaN                NaN                               NaN
85                                                 NaN                      NaN                          NaN                               NaN                                          NaN                                         NaN                NaN                               NaN
86                                                 NaN                      NaN                          NaN                               NaN                                          NaN                                         NaN                NaN                               NaN
87                                             Players    Nombre de point total 

---
# Pipeline RAG bout-en-bout

On exécute le pipeline complet (recherche → contexte → prompt → réponse Mistral) sur quatre cas : question textuelle (cas favorable), question **chiffrée** (cas limite), question **hors-sujet** (garde-fou), et question **bruitée** (robustesse).

In [17]:
import ast
from pathlib import Path

def load_system_prompt_from_app(app_file="MistralChat.py", var_name="SYSTEM_PROMPT"):
    """Extrait la chaîne SYSTEM_PROMPT du code de l'app, sans exécuter Streamlit."""
    tree = ast.parse(Path(app_file).read_text(encoding="utf-8"))
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign) and any(
            isinstance(t, ast.Name) and t.id == var_name for t in node.targets
        ):
            value = node.value
            # Chaîne simple
            if isinstance(value, ast.Constant) and isinstance(value.value, str):
                return value.value
            # f-string sans interpolation dynamique (l'app utilise {{...}} = accolades littérales)
            if isinstance(value, ast.JoinedStr):
                if all(isinstance(p, ast.Constant) for p in value.values):
                    return "".join(p.value for p in value.values)
                raise ValueError("SYSTEM_PROMPT contient une interpolation dynamique inattendue.")
    raise ValueError(f"{var_name} introuvable dans {app_file}")


SYSTEM_PROMPT = load_system_prompt_from_app()
print(f"SYSTEM_PROMPT importé depuis MistralChat.py ({len(SYSTEM_PROMPT)} caractères).")


SYSTEM_PROMPT importé depuis MistralChat.py (226 caractères).


In [18]:
def run_rag(question, k=SEARCH_K):
    """Exécute le pipeline RAG complet et retourne la réponse + méta."""
    # 1. Recherche
    results = mgr.search(question, k=k)

    # 2. Construction du contexte
    if results:
        context_str = "\n\n---\n\n".join([
            f"Source: {r['metadata'].get('source', '?')} (Score: {r['score']:.1f}%)\nContenu: {r['text']}"
            for r in results
        ])
    else:
        context_str = "Aucune information pertinente trouvée dans la base de connaissances."

    # 3. Prompt
    prompt = SYSTEM_PROMPT.format(context_str=context_str, question=question)

    # 4. Appel LLM
    response = client.chat(
        model=MODEL_NAME,
        messages=[ChatMessage(role="user", content=prompt)],
        temperature=0.1
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [(r['metadata'].get('source','?'), r['score']) for r in results],
        "nb_chunks": len(results),
        "tokens": response.usage.total_tokens,
    }

## Question chiffrée simple : meilleur 3P%

In [19]:
q1 = "Quel joueur a le meilleur pourcentage à 3 points ?"
print(f"  ❓ {q1}")
print()

result = run_rag(q1)
ok(f"Réponse reçue ({result['tokens']} tokens, {result['nb_chunks']} chunks utilisés)")
print()
print("  RÉPONSE :")
for line in result['answer'].split('\n'):
    print(f"    {line}")
print()
print("  SOURCES :")
for src, score in result['sources']:
    print(f"    • {score:.1f}%  {src}")

2026-06-04 12:17:58,037 - INFO - Recherche des 5 chunks les plus pertinents pour: 'Quel joueur a le meilleur pourcentage à 3 points ?'


  ❓ Quel joueur a le meilleur pourcentage à 3 points ?



2026-06-04 12:17:58,257 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:17:58,259 - INFO - 5 chunks pertinents trouvés.
2026-06-04 12:18:01,206 - INFO - HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


 ✅ Réponse reçue (1759 tokens, 5 chunks utilisés)

  RÉPONSE :
    D'après les données disponibles dans le fichier **regular NBA.xlsx** (Feuille: *Dictionnaire des données*), le joueur avec le **meilleur pourcentage à 3 points (3P%)** est **Shai Gilgeous-Alexander**, avec un taux de réussite de **37,5%** (source : Feuille *Analyse*, ligne 88).
    
    ### Contexte supplémentaire :
    - **Shai Gilgeous-Alexander** est également en tête du classement des **points totaux** (2485) dans cette analyse.
    - Son **efficacité globale** est soulignée par un **True Shooting % (TS%)** élevé (non précisé ici, mais souvent corrélé à une bonne réussite à 3 points).
    - Les débats sur Reddit montrent que l'efficacité pure (comme le 3P%) est parfois critiquée si elle ne s'accompagne pas de succès en playoffs (ex. : Reggie Miller, souvent cité comme "le plus efficace sans titre").
    
    ### Pour aller plus loin :
    Si vous souhaitez une analyse plus large (ex. : joueurs actifs avec un 3P% > 4

### Limite observée sur le run actuel

Même lorsque la réponse cite une source existante, elle peut rester incorrecte.

Ici, le modèle répond **Shai Gilgeous-Alexander — 37,5 %** en s'appuyant sur un chunk de la feuille `Analyse`, mais il ne calcule pas réellement le maximum de la colonne `3P%` sur l'ensemble du fichier Excel.

Dans l'extrait affiché plus haut, **Nikola Jokić** a déjà un `3P%` de **41,7 %**, supérieur à 37,5 %. Cela montre que le RAG seul reformule un contexte récupéré, mais ne réalise pas une agrégation fiable sur les données tabulaires.

Cette limite justifie l'ajout ultérieur d'une base SQLite et d'un SQL Tool pour les questions chiffrées.

### 🔬 Preuve d'un ancien run Streamlit : « Steve Nash » sort-il vraiment des données ?

Lors d'un test manuel précédent dans l'application Streamlit, le modèle a répondu :

> *« D'après le fichier **regular NBA.xlsx** (Dictionnaire des données), le joueur avec le meilleur 3P% est **Steve Nash – 42,8 %** »*

Cette réponse ne correspond pas au run actuel du notebook, qui répond Shai Gilgeous-Alexander. Elle reste utile comme preuve d'un risque important : le modèle peut produire une réponse plausible mais non ancrée dans les sources.

Vérifions si Steve Nash figure réellement dans les données indexées — sachant qu'il est à la retraite depuis 2015.

In [20]:
import pickle
from utils.config import DOCUMENT_CHUNKS_FILE

chunks_all = pickle.load(open(DOCUMENT_CHUNKS_FILE, "rb"))
all_text = "\n".join(c["text"] for c in chunks_all).lower()
xls_text = "\n".join(c["text"] for c in chunks_all if ".xlsx" in c["metadata"].get("source", "")).lower()
ctx = " ".join(r["text"] for r in mgr.search(q1, k=SEARCH_K)).lower()

print("  1) Steve Nash dans les données indexées ?")
print(f"       'steve nash' (toutes sources) : {all_text.count('steve nash')}")
print(f"       'nash' dans l'Excel NBA       : {xls_text.count('nash')}")
print(f"       'nash' ailleurs               : {all_text.count('nash')}  (commentaires Reddit, jamais une stat)")

print("\n  2) Steve Nash dans le contexte récupéré pour cette question ?")
print(f"       'nash' dans le contexte : {'nash' in ctx}")

print()
if all_text.count("steve nash") == 0 and "nash" not in ctx:
    ko("« Steve Nash - 42,8 % » = HALLUCINATION : absent des données ET du contexte récupéré.")
    ko("Le LLM l'invente depuis sa mémoire et l'attribue faussement à regular NBA.xlsx.")
else:
    ok("Steve Nash présent dans les données (à réexaminer).")

2026-06-04 12:18:01,242 - INFO - Recherche des 5 chunks les plus pertinents pour: 'Quel joueur a le meilleur pourcentage à 3 points ?'
2026-06-04 12:18:01,471 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:18:01,477 - INFO - 5 chunks pertinents trouvés.


  1) Steve Nash dans les données indexées ?
       'steve nash' (toutes sources) : 0
       'nash' dans l'Excel NBA       : 0
       'nash' ailleurs               : 4  (commentaires Reddit, jamais une stat)

  2) Steve Nash dans le contexte récupéré pour cette question ?
       'nash' dans le contexte : False

 ❌ « Steve Nash - 42,8 % » = HALLUCINATION : absent des données ET du contexte récupéré.
 ❌ Le LLM l'invente depuis sa mémoire et l'attribue faussement à regular NBA.xlsx.


**Verdict.** « Steve Nash – 42,8 % » est une **hallucination** observée lors d'un test manuel précédent :

- Steve Nash n'apparaît pas dans les données Excel indexées.
- Les seules mentions de « Nash » proviennent de commentaires Reddit et ne correspondent pas à une statistique du fichier NBA.
- Le modèle attribue donc faussement une information externe au fichier `regular NBA.xlsx`.

Cette réponse est plausible dans le monde réel, car Steve Nash a bien eu un très bon pourcentage à 3 points en carrière, mais elle n'est **pas fondée sur les sources du projet**. C'est exactement le type d'erreur que la métrique **faithfulness** de RAGAS doit aider à mesurer.

## Question chiffrée complexe : rebonds domicile / extérieur

In [24]:
q2 = "Compare les statistiques de rebonds à domicile et à l'extérieur sur les 5 derniers matchs"
print(f"  ❓ {q2}")
print()

result2 = run_rag(q2)
answer = result2['answer']

print()
print("  RÉPONSE :")
for line in answer.split('\n'):
    print(f"    {line}")

2026-06-04 12:18:55,119 - INFO - Recherche des 5 chunks les plus pertinents pour: 'Compare les statistiques de rebonds à domicile et à l'extérieur sur les 5 derniers matchs'


  ❓ Compare les statistiques de rebonds à domicile et à l'extérieur sur les 5 derniers matchs



2026-06-04 12:18:55,482 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:18:55,502 - INFO - 5 chunks pertinents trouvés.
2026-06-04 12:18:59,863 - INFO - HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"



  RÉPONSE :
    **Analyse des statistiques de rebonds à domicile vs. à l'extérieur (5 derniers matchs)**
    
    Pour comparer les performances de rebonds des joueurs à domicile et à l'extérieur, nous devons nous concentrer sur les indicateurs clés suivants :
    - **REB** (Rebonds totaux par match)
    - **OREB** (Rebonds offensifs)
    - **DREB** (Rebonds défensifs)
    - **REB%** (Pourcentage de rebonds captés parmi ceux disponibles)
    
    ### **Facteurs influençant les différences domicile/extérieur**
    1. **Fatigue des déplacements** : Les matchs à l'extérieur peuvent être plus éprouvants physiquement, réduisant l'énergie pour les rebonds.
    2. **Ambiance du public** : Les salles à domicile offrent souvent un avantage psychologique, poussant les joueurs à plus d'agressivité.
    3. **Style de jeu des adversaires** : Certaines équipes ont des systèmes défensifs plus agressifs à l'extérieur.
    
    ### **Exemple d'analyse (si données disponibles)**
    Supposons que nous 

⚠️ **Réponse non ancrée (*faithfulness*).** Le modèle répond avec une analyse générique alors que les données disponibles ne contiennent pas les rebonds domicile / extérieur sur les 5 derniers matchs.

La réponse mentionne des exemples hypothétiques et des joueurs comme Jokic, Giannis, Sabonis ou Embiid, mais ces éléments ne sont pas calculés depuis les sources du projet. C'est un cas clair où le modèle devrait reconnaître que les données sont insuffisantes.

## Question hors sujet

In [25]:
q3 = "Quelle est la recette de la ratatouille ?"
print(f"  ❓ {q3}")
print()

result3 = run_rag(q3)
answer3 = result3['answer']

print()
print("  RÉPONSE :")
for line in answer3[:400].split('\n'):
    print(f"    {line}")

print()
print("Sources retournées (même pour une question hors-sujet, FAISS renvoie des résultats) :")
for src, score in result3['sources']:
    print(f"    • {score:.1f}%  {src}")

2026-06-04 12:20:11,263 - INFO - Recherche des 5 chunks les plus pertinents pour: 'Quelle est la recette de la ratatouille ?'


  ❓ Quelle est la recette de la ratatouille ?



2026-06-04 12:20:11,680 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:20:11,700 - INFO - 5 chunks pertinents trouvés.
2026-06-04 12:20:13,263 - INFO - HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"



  RÉPONSE :
    **NBA Analyst AI** 🏀🔥 :
    
    *"Désolé, mais je suis ici pour décortiquer les stats NBA, pas pour cuisiner ! 😅*
    
    *Si tu veux analyser les performances des équipes comme les Grizzlies (MEM) ou les joueurs comme Ja Morant, je suis ton homme. Sinon, direction un bon livre de recettes !*
    
    *Mais si tu veux un débat NBA, dis-moi :*
    - *Qui est le MVP actuel ?*
    - *Les Nuggets peuvent-ils défendre leur titre ?*

Sources retournées (même pour une question hors-sujet, FAISS renvoie des résultats) :
    • 74.2%  regular NBA.xlsx (Feuille: Dictionnaire des données)
    • 73.9%  regular NBA.xlsx (Feuille: Dictionnaire des données)
    • 73.4%  regular NBA.xlsx (Feuille: Analyse Vide)
    • 73.2%  regular NBA.xlsx (Feuille: Analyse Vide)
    • 73.2%  regular NBA.xlsx (Feuille: Analyse)


 ✅ Le LLM refuse de répondre hors-sujet (mentionne le sujet pour le rejeter = comportement correct)

## Question bruitée / mal formulée

In [26]:
q4 = "kl vs okc stts rebnd lst 5 gm??"
print(f"  ❓ {q4}")
print()

result4 = run_rag(q4)
ok(f"Pipeline n'a pas planté ({result4['nb_chunks']} chunks, {result4['tokens']} tokens)")
print()
print("  RÉPONSE :")
for line in result4['answer'][:400].split('\n'):
    print(f"    {line}")

2026-06-04 12:20:21,109 - INFO - Recherche des 5 chunks les plus pertinents pour: 'kl vs okc stts rebnd lst 5 gm??'


  ❓ kl vs okc stts rebnd lst 5 gm??



2026-06-04 12:20:21,477 - INFO - HTTP Request: POST https://api.mistral.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-06-04 12:20:21,529 - INFO - 5 chunks pertinents trouvés.
2026-06-04 12:20:25,291 - INFO - HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"


 ✅ Pipeline n'a pas planté (5 chunks, 1461 tokens)

  RÉPONSE :
    **NBA Analyst AI** : *"Kl vs OKC - Stats et rebond sur les 5 derniers matchs ?"*
    
    D'après les données disponibles dans le fichier **regular NBA.xlsx**, voici les statistiques clés des **5 derniers matchs** entre les **Los Angeles Clippers (LAL)** et les **Oklahoma City Thunder (OKC)** :
    
    ### **Statistiques des 5 derniers matchs (LAL vs OKC)**
    *(Si les données ne sont pas disponibles, voici une ana
